## Chapter 7 – Running on Quantum Hardware

This notebook moves beyond simulation and begins working directly with live IBM Quantum hardware using Qiskit Runtime. The examples connect to an active quantum backend, inspect coherence data, run and transpile circuits for physical hardware, compare simulator and hardware results, experiment with Bell-state entanglement, and collect system-level profiling metrics.

The examples build on one another throughout the notebook. Later cells reuse the selected backend, calibration data, circuits, transpiled results, and measurement data created earlier.

### Notebook Setup

Before running the examples, execute **Code 7-0** to install the required Qiskit Runtime packages.

You will also need:

- An IBMid account
- An IBM Quantum API key
- A Google Colab Secret labeled `IBMQ`

> **Important:** Run the notebook cells sequentially from top to bottom. Later examples depend on variables and results created by earlier cells.

> **Tip:** If the Colab runtime disconnects or times out, reconnect, rerun **Code 7-0**, and restart the notebook from the beginning.

> **Note:** These examples use shared quantum hardware. Backend availability, queue times, calibration values, and execution limits may change between runs.

### Code 7-0: Installing Qiskit Runtime

Install the IBM Quantum Runtime components used throughout this chapter, including the `qiskit-ibm-runtime` package for accessing live quantum hardware and `qiskit-aer` for local simulation support.

In [11]:
!pip install qiskit qiskit-ibm-runtime qiskit-aer pylatexenc --quiet

### Code 7-1: Connecting to an IBM Quantum System

This example connects the notebook to IBM Quantum using Qiskit Runtime. Before running it, complete the `pip install` setup in Code 7-0, create an IBMid account, generate an IBM Quantum API key, and store the key in Google Colab Secrets using the label `IBMQ`.

The code then initializes the runtime service, retrieves the available backends, and automatically selects an operational quantum processor.

In [12]:
# === Code 7-1: Connecting to an IBM Quantum System ===

# --- Step 1: Load Your API Key ---
# In Google Colab, click the key-shaped Secrets icon
# and create a secret named "IBMQ".

from google.colab import userdata

# Load the API key from Colab Secrets
IBMQ_API_KEY = userdata.get('IBMQ')

print("API key retrieved successfully.")


# --- Step 2: Configure Qiskit Runtime ---
# Initialize the Qiskit Runtime connection.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",  # IBM Quantum service
    token=IBMQ_API_KEY               # Stored API key
)

print("Connected to IBM Quantum.")


# --- Step 3: Select a Quantum Backend ---
# Retrieve the systems available to your account.

available_systems = service.backends()

print(f"Available backends: {len(available_systems)}")

for backend in available_systems:
    print(backend.name)

# Automatically select an operational backend.

target_backend = service.least_busy(
    min_num_qubits=5,  # Require at least 5 qubits
    operational=True   # Only active systems
)

print(f"Selected backend: {target_backend.name}")

qiskit_runtime_service._discover_account:WARNING:2026-09-29 01:22:22,019: Loading account with the given token. A saved account will not be used.


API key retrieved successfully.


qiskit_runtime_service.__init__:WARNING:2026-09-29 01:22:25,742: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-09-29 01:22:25,743: Loading instance: open-instance, plan: open


Connected to IBM Quantum.
Available backends: 3
ibm_fez
ibm_kingston
ibm_marrakesh


qiskit_runtime_service.backends:WARNING:2026-09-29 01:22:27,925: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-09-29 01:22:29,385: Using instance: open-instance, plan: open


Selected backend: ibm_fez


### Code 7-2: Observing Qubit Coherence Times

This section retrieves live calibration telemetry from the selected IBM Quantum backend and visualizes the processor’s `T1` and `T2` coherence measurements. The notebook calculates median coherence values, identifies valid qubit readings, and plots stability trends across the processor topology. These measurements help reveal how long fragile quantum states can remain reliable before decoherence begins influencing computation quality.

In [ ]:
# === Code 7-2: Inspecting Qubit Coherence Times ===

import pandas as pd
import matplotlib.pyplot as plt


def get_coherence_values(props):
    """Return qubits with valid T1 and T2 measurements."""
    qubits = []
    t1_values = []
    t2_values = []

    for q in range(len(props.qubits)):
        try:
            t1 = props.t1(q) * 1e6
            t2 = props.t2(q) * 1e6
            qubits.append(q)
            t1_values.append(t1)
            t2_values.append(t2)
        except Exception:
            pass

    return qubits, t1_values, t2_values


def plot_coherence_times(
    qubits, t1_values, t2_values, backend_name, window=7
):
    """Plot T1 and T2 coherence measurements."""
    df = pd.DataFrame({
        "qubit": qubits,
        "T1": t1_values,
        "T2": t2_values
    })

    df["T1 trend"] = df["T1"].rolling(window, center=True).mean()
    df["T2 trend"] = df["T2"].rolling(window, center=True).mean()

    # Shade the middle 60% of measured coherence times.
    all_times = pd.concat([df["T1"], df["T2"]])
    range_low = all_times.quantile(0.20)
    range_high = all_times.quantile(0.80)

    plt.figure(figsize=(10, 5))
    plt.axhspan(
        range_low, range_high, alpha=0.10, label="Typical range"
    )
    plt.scatter(
        df["qubit"], df["T1"], s=18, alpha=0.25, label="T1"
    )
    plt.scatter(
        df["qubit"], df["T2"], s=18, alpha=0.25, label="T2"
    )
    plt.plot(
        df["qubit"], df["T1 trend"], linewidth=2, label="T1 trend"
    )
    plt.plot(
        df["qubit"], df["T2 trend"], linewidth=2, label="T2 trend"
    )

    plt.title(f"Coherence Times for {backend_name}")
    plt.xlabel("Qubit Index")
    plt.ylabel("Coherence Time (microseconds)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


# Read current backend calibration properties.
props = target_backend.properties()
backend_name = target_backend.name

# Retrieve qubits with valid T1 and T2 measurements.
qubits, t1_values, t2_values = get_coherence_values(props)

# Plot raw measurements and smoothed trends.
plot_coherence_times(
    qubits, t1_values, t2_values, backend_name
)

### Code 7-3: Running a Quantum Die on Hardware

This example executes the quantum die experiment from Chapter 3 on a live IBM Quantum processor. The logical circuit is transpiled for the selected backend, submitted through Qiskit Runtime, and executed using repeated measurement shots. The returned bitstrings are then mapped onto die faces, exposing how physical hardware behavior differs slightly from the idealized simulator environment.

In [15]:
# === Code 7-3: Running a Quantum Die on Hardware ===

from collections import Counter

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.executor_sampler import Sampler


# Step 1: Build the quantum die circuit.
qc = QuantumCircuit(3, 3)
qc.h([0, 1, 2])
qc.measure([0, 1, 2], [0, 1, 2])

# Step 2: Adapt the circuit to the backend hardware.
pm = generate_preset_pass_manager(
    backend=target_backend,
    optimization_level=1
)

# Transpile the logical circuit for the selected backend
isa_qc = pm.run(qc)

# Step 3: Run the circuit on the quantum processor.
sampler = Sampler(mode=target_backend)

job = sampler.run([isa_qc], shots=2000)

print(f"Job ID: {job.job_id()}")

result = job.result()

counts = result[0].data.c.get_counts()

# Step 4: Map bitstrings to die faces.

# Map six bitstrings onto die faces.
mapping = {
    "000": 1, "001": 2, "010": 3,
    "011": 4, "100": 5, "101": 6
}

die_counts = Counter()

for bits, count in counts.items():
    if bits in mapping:
        die_counts[mapping[bits]] += count

unused = counts.get("110", 0) + counts.get("111", 0)

print("Quantum die results:",
      dict(sorted(die_counts.items())))

print("Unused outcomes:", unused)

qiskit_runtime_service.backends:WARNING:2026-09-29 01:25:50,904: Using instance: open-instance, plan: open


Job ID: dath785vr3kc73elgisg
Quantum die results: {1: 260, 2: 255, 3: 270, 4: 259, 5: 235, 6: 251}
Unused outcomes: 470


### Code 7-4: Inspecting a Transpiled Circuit

This section compares a logical quantum circuit against its transpiled hardware-aware form. The notebook demonstrates how Qiskit remaps logical qubits, rewrites operations into backend-native gates, and adjusts circuit depth to satisfy physical hardware constraints. Displaying both circuits side by side helps expose the transition from abstract quantum programming to physically executable processor instructions.

In [ ]:
# === Code 7-4: Inspecting a Transpiled Circuit ===

from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager

# Step 1: Build a small logical circuit.
logical_qc = QuantumCircuit(4, 4)
logical_qc.h(0)
logical_qc.cx(0, 3)
logical_qc.measure(range(4), range(4))

print("Original circuit depth:", logical_qc.depth())

# Step 2: Transpile for the selected backend.
pm = generate_preset_pass_manager(
    backend=target_backend,
    optimization_level=1
)

transpiled_qc = pm.run(logical_qc)

print("Transpiled circuit depth:", transpiled_qc.depth())

# Step 3: Display both circuits.
display(logical_qc.draw("mpl"))
display(transpiled_qc.draw("mpl", idle_wires=False))

### Code 7-5: Creating a Bell State

This example builds a simple Bell-state circuit and executes it on an idealized simulator. A Hadamard gate places the first qubit into superposition, while a controlled-X operation creates entanglement between the two qubits. Repeated measurements reveal the correlated `00` and `11` states characteristic of Bell-state behavior in a mathematically clean simulation environment.

In [ ]:
# === Code 7-5: Creating a Bell State ===

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Step 1: Build a Bell-state circuit.
bell_qc = QuantumCircuit(2, 2)
bell_qc.h(0)           # Place q0 into superposition.
bell_qc.cx(0, 1)       # Use a CNOT gate to entangle q1 with q0.
bell_qc.measure([0, 1], [0, 1])

# Step 2: Run on a simulator.
sim = AerSimulator()

sim_counts = sim.run(
    bell_qc,
    shots=2000
).result().get_counts()

print("Bell-state results:", sim_counts)

# Step 3: Draw the circuit.
bell_qc.draw("mpl")

### Code 7-6: Running a Bell State on Hardware

This section executes the Bell-state circuit from Code 7-5 on a live quantum processor. After transpilation, the circuit is submitted through Qiskit Runtime and compared directly against the simulator baseline. The returned measurements let us see how well the physical processor preserves the relationship between the two entangled qubits.

In [ ]:
# === Code 7-6: Running a Bell State on Hardware ===

import matplotlib.pyplot as plt
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Helper used to compare simulator and hardware measurements.
def plot_bell_comparison(sim_counts, hw_counts):
    states = ["00", "01", "10", "11"]
    colors = ["#2F5D8C", "#C44E00", "#C44E00", "#4C9F70"]
    sim = [sim_counts.get(s, 0) for s in states]
    hw = [hw_counts.get(s, 0) for s in states]
    ymax = max(sim + hw) * 1.15

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, vals, title in zip(axes, [sim, hw],
                               ["Simulator Results", "Hardware Results"]):
        bars = ax.bar(states, vals, color=colors)
        ax.set(title=title, xlabel="Measured State", ylim=(0, ymax))
        ax.bar_label(bars, padding=3)
    axes[0].set_ylabel("Counts")
    plt.tight_layout()
    plt.show()

# Step 1: Transpile for the selected backend.
pm = generate_preset_pass_manager(
    backend=target_backend, optimization_level=1
)
hardware_qc = pm.run(bell_qc)

# Step 2: Submit the circuit to hardware.
sampler = Sampler(mode=target_backend)
job = sampler.run([hardware_qc], shots=2000)
print(f"Job ID: {job.job_id()}")

hardware_counts = job.result()[0].data.c.get_counts()
print("Hardware Bell-state results:", hardware_counts)

# Step 3: Check how often the Bell relationship held.
matches = hardware_counts.get("00", 0) + hardware_counts.get("11", 0)
total = sum(hardware_counts.values())
print(f"Matching outcomes: {matches}/{total} ({100 * matches / total:.1f}%)")

# Step 4: Display the transpiled circuit.
display(hardware_qc.draw("mpl", idle_wires=False))

# Step 5: Compare simulator and hardware results.
plot_bell_comparison(sim_counts, hardware_counts)

### Code 7-7: Profiling a Quantum Hardware Run

This section builds a unified hardware telemetry dashboard by combining results from Code 7-2, Code 7-5, and Code 7-6. The notebook reuses the live coherence measurements stored in `coherence_df`, the Bell-state circuit stored in `bell_qc`, and the transpiled hardware execution results stored in `hardware_qc` and `hardware_counts`. The dashboard then combines coherence metrics, transpilation statistics, circuit depth expansion, measurement mismatch analysis, and scaling estimates into a compact systems-level profile. The goal is to expose the physical engineering signals underneath a live quantum execution and connect them directly to the behavior of an actual quantum processor.

In [ ]:
# === Code 7-7: Profiling a Quantum Hardware Run ===
# Requires: coherence_df (Code 7-2), bell_qc (Code 7-5),
# hardware_qc/hardware_counts (Code 7-6), and target_backend.

import matplotlib.pyplot as plt

# --- Adjustable dashboard settings ---
FIG_SIZE = (14, 10)
GRID_RATIOS = [0.72, 0.72, 0.72, 0.18, 1.45]

LEFT, RIGHT, TOP, BOTTOM = 0.07, 0.97, 0.90, 0.08
WSPACE, HSPACE = 0.32, 0.14

CARD_COLOR = "#F7F7F7"
CARD_EDGE = "#D8D8D8"

COLOR_INFO = "#1F77B4"
COLOR_STABLE = "#2E7D32"
COLOR_WARN = "#D55E00"
COLOR_SCALE = "#6A3D9A"

COLOR_LOGICAL = "#9ECAE1"
COLOR_HARDWARE = "#F28E2B"
COLOR_00 = "#2F5D8C"
COLOR_11 = "#4C9F70"
COLOR_MISMATCH = "#C44E00"

MAIN_TITLE_SIZE = 22
CARD_VALUE_SIZE = 20
CARD_LABEL_SIZE = 15
CHART_TITLE_SIZE = 16
AXIS_LABEL_SIZE = 13
TICK_LABEL_SIZE = 12
BAR_LABEL_SIZE = 12

CARD_NUMBER_SIZE = 12
CARD_NUMBER_COLOR = "#777777"

VALUE_Y, LABEL_Y = 0.62, 0.27
NUMBER_X, NUMBER_Y = 0.06, 0.92


def draw_metric_card(ax, number, label, value, accent):
    """Draw one dashboard metric card."""
    ax.set_facecolor(CARD_COLOR)
    ax.text(NUMBER_X, NUMBER_Y, str(number), ha="left", va="top",
            fontsize=CARD_NUMBER_SIZE, fontweight="bold",
            color=CARD_NUMBER_COLOR)
    ax.text(0.5, VALUE_Y, str(value), ha="center", va="center",
            fontsize=CARD_VALUE_SIZE, fontweight="bold", color=accent)
    ax.text(0.5, LABEL_Y, label, ha="center", va="center",
            fontsize=CARD_LABEL_SIZE)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_color(CARD_EDGE)


def label_bars_horizontal(ax, bars):
    """Label horizontal bars."""
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.15, bar.get_y() + bar.get_height() / 2,
                f"{int(width)}", va="center", fontsize=BAR_LABEL_SIZE)


def label_bars_vertical(ax, bars):
    """Label vertical bars."""
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, height + 20,
                f"{int(height)}", ha="center", fontsize=BAR_LABEL_SIZE)


def plot_hardware_dashboard(metrics, counts):
    """Display coherence, topology, depth, mismatches, and scaling."""
    states = ["00", "11", "01", "10"]
    values = [counts.get(state, 0) for state in states]

    fig = plt.figure(figsize=FIG_SIZE)
    gs = fig.add_gridspec(5, 4, height_ratios=GRID_RATIOS)

    fig.suptitle(f"Quantum Hardware Profile: {metrics['backend']}",
                 fontsize=MAIN_TITLE_SIZE, fontweight="bold")

    cards = [
        ("Physical qubits", metrics["physical_qubits"], COLOR_INFO),
        ("Coupling paths", metrics["coupling_paths"], COLOR_INFO),
        ("Median T1", f"{metrics['median_t1']:.1f} µs", COLOR_STABLE),
        ("Median T2", f"{metrics['median_t2']:.1f} µs", COLOR_STABLE),
        ("CX after transpile", metrics["cnot_gates"], COLOR_WARN),
        ("Depth expansion", metrics["depth_expansion"], COLOR_WARN),
        ("Mismatch rate", f"{metrics['mismatch_rate']:.2f}%", COLOR_WARN),
        ("Mismatched", metrics["mismatched"], COLOR_WARN),
        ("Estimated logical", f"{metrics['estimated_logical']:.4f}",
         COLOR_SCALE),
        ("Target logical", metrics["target_logical"], COLOR_SCALE),
        ("Physical needed", metrics["required_physical"], COLOR_SCALE),
        ("Total shots", metrics["total_shots"], COLOR_SCALE),
    ]

    for i, (label, value, accent) in enumerate(cards, start=1):
        row, col = divmod(i - 1, 4)
        draw_metric_card(fig.add_subplot(gs[row, col]),
                         i, label, value, accent)

    ax1 = fig.add_subplot(gs[4, :2])
    depth_values = [metrics["original_depth"], metrics["hardware_depth"]]
    bars1 = ax1.barh(["Logical", "Hardware"], depth_values,
                     color=[COLOR_LOGICAL, COLOR_HARDWARE],
                     edgecolor="#444444", hatch=["", "//"])
    ax1.set_title("Circuit Depth", fontsize=CHART_TITLE_SIZE, pad=8)
    ax1.set_xlabel("Depth", fontsize=AXIS_LABEL_SIZE)
    ax1.set_xlim(0, max(depth_values) * 1.1)
    ax1.tick_params(labelsize=TICK_LABEL_SIZE)
    ax1.grid(axis="x", alpha=0.25)
    label_bars_horizontal(ax1, bars1)

    ax2 = fig.add_subplot(gs[4, 2:])
    bars2 = ax2.bar(
        states, values,
        color=[COLOR_00, COLOR_11, COLOR_MISMATCH, COLOR_MISMATCH],
        edgecolor="#444444", hatch=["", "", "//", "//"]
    )
    ax2.set_title("Bell-State Measurements",
                  fontsize=CHART_TITLE_SIZE, pad=8)
    ax2.set_xlabel("Measured Bitstring", fontsize=AXIS_LABEL_SIZE,
                   labelpad=8)
    ax2.set_ylabel("Counts", fontsize=AXIS_LABEL_SIZE)
    ax2.set_ylim(0, max(values) * 1.25)
    ax2.tick_params(labelsize=TICK_LABEL_SIZE)
    ax2.grid(axis="y", alpha=0.25)
    label_bars_vertical(ax2, bars2)

    plt.subplots_adjust(left=LEFT, right=RIGHT, top=TOP, bottom=BOTTOM,
                        wspace=WSPACE, hspace=HSPACE)
    plt.show()


# --- Support metrics ---
coupling_paths = len(target_backend.coupling_map.get_edges())
original_depth = bell_qc.depth()
hardware_depth = hardware_qc.depth()
cnot_gates = hardware_qc.count_ops().get("cx", 0)
total_shots = sum(hardware_counts.values())

# Simplified educational scaling assumptions.
correction_ratio = 1000
target_logical = 50

# --- System and coherence telemetry ---
physical_qubits = target_backend.num_qubits
median_t1 = coherence_df["T1"].median()
median_t2 = coherence_df["T2"].median()

# --- Execution telemetry ---
depth_expansion = hardware_depth - original_depth
mismatched = hardware_counts.get("01", 0) + hardware_counts.get("10", 0)
mismatch_rate = 100 * mismatched / total_shots

# --- Scaling estimates ---
estimated_logical = physical_qubits / correction_ratio
required_physical = target_logical * correction_ratio

metrics = {
    "backend": target_backend.name,
    "physical_qubits": physical_qubits,
    "coupling_paths": coupling_paths,
    "median_t1": median_t1,
    "median_t2": median_t2,
    "original_depth": original_depth,
    "hardware_depth": hardware_depth,
    "depth_expansion": depth_expansion,
    "cnot_gates": cnot_gates,
    "total_shots": total_shots,
    "mismatched": mismatched,
    "mismatch_rate": mismatch_rate,
    "estimated_logical": estimated_logical,
    "target_logical": target_logical,
    "required_physical": required_physical,
}

plot_hardware_dashboard(metrics, hardware_counts)